In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import glob
import os

data_dir = '../data_out/'
stats_dir = '../statistics/'

In [2]:
def get_df_from_nc(ds, location, var, from_date, to_date):
    ds = ds.sel(location=location,time=slice(from_date, to_date))
    time_dim = ds[var].dims[0]
    z_mc = ds['z_mc'].values
    height_values = z_mc - (z_mc[-1] - 5)
    df = pd.DataFrame(ds[var].values.T, index=height_values, columns=pd.to_datetime(ds[time_dim],utc=True))
    return df

def calculate_metrics(df_obs, df_sim):
    # Align observations and simulations
    obs, sim = df_obs.align(df_sim, join="inner", axis=None)

    # Convert to arrays and retain only paired finite values
    obs = obs.to_numpy(dtype=float).ravel()
    sim = sim.to_numpy(dtype=float).ravel()

    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    n = len(obs)

    if n == 0:
        return {
            "N": 0,
            "R2": np.nan,
            "MAE": np.nan,
            "MBE": np.nan,
            "RMSE": np.nan,
        }

    error = sim - obs

    return {
        "N": str(int(n)),
        "R2": "{:.2f}".format(
            np.corrcoef(obs, sim)[0, 1] ** 2
            if n >= 2 and np.std(obs) > 0 and np.std(sim) > 0
            else np.nan
        ),
        "MAE": "{:.2f}".format(np.mean(np.abs(error))),
        "MBE": "{:.2f}".format(np.mean(error)),
        "RMSE": "{:.2f}".format(np.sqrt(np.mean(error**2))),
    }

## Script settings

In [3]:
# location settings
iop, sel_loc = 'IOP1', ['Q255', 'Q267', 'Q267L', 'Q272', 'Q272L', 'Q272R', 'F309']
# iop, sel_loc = 'IOP2', ['Q257', 'Q272', 'Q293']

var = 'theta'

In [4]:
# time settings
if iop == 'IOP1':
    from_date, to_date = '2025-08-07', '2025-08-10'
elif iop == 'IOP2':
    from_date, to_date = '2025-08-18', '2025-08-19'
else:
    raise ValueError(f"Unknown IOP: {iop}")

## Get data

The data used for this figure is available at Zenodo:

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.19596612.svg)](https://doi.org/10.5281/zenodo.19596612)

In [5]:
# load data
files = glob.glob(data_dir + f'icon/HEFEX3__ICON_v370_2030_v2_uav_locations_50ml__1h_{iop}*.nc')
if len(files) == 1:
    ds_icon = xr.open_dataset(files[0])
else:
    raise ValueError(f"Expected one file for ICON for {iop}, found {len(files)} files.")

files = glob.glob(data_dir + f'obs/HEFEX3__Obs_UAV_profiles_L2__1h_{iop}_*.nc')
if len(files) == 1:
    ds_uav = xr.open_dataset(files[0])
else:
    raise ValueError(f"Expected one file for Obs for {iop}, found {len(files)} files.")

In [6]:
df = pd.DataFrame()

for loc in sel_loc:
    df_icon = get_df_from_nc(ds_icon, loc, var, from_date, to_date)
    df_uav  = get_df_from_nc(ds_uav, loc, f'{var}_ml', from_date, to_date)

    metrics = calculate_metrics(df_uav, df_icon)
    df = pd.concat([df,pd.Series(metrics, name=loc)], axis=1)

ds_icon.close()
ds_uav.close()

df

,Q255,Q267,Q267L,Q272,Q272L,Q272R,F309
N,499,314,211,565,184,222,464
R2,0.83,0.72,0.73,0.83,0.72,0.81,0.85
MAE,0.84,0.84,0.91,0.82,0.91,0.94,0.85
MBE,-0.23,-0.23,-0.20,-0.32,-0.14,-0.43,-0.74
RMSE,1.12,1.03,1.07,1.10,1.18,1.12,1.05


In [7]:
out_file = stats_dir + f'HEFEX3__UAV_ICON_stats__{iop}_{var}.csv'
df.to_csv(out_file)
print('Saved stats to', out_file)

Saved stats to ../statistics/HEFEX3__UAV_ICON_stats__IOP1_theta.csv


## Merge IOP1 and IOP2 stats into one table for display in paper

In [8]:
iop1_file = stats_dir + f'HEFEX3__UAV_ICON_stats__IOP1_{var}.csv'
iop2_file = stats_dir + f'HEFEX3__UAV_ICON_stats__IOP2_{var}.csv'

if os.path.isfile(iop1_file) and os.path.isfile(iop2_file):
    print('Both files exist, proceeding with merge.')

    df_iop1 = pd.read_csv(iop1_file, index_col=0, dtype=str)
    df_iop2 = pd.read_csv(iop2_file, index_col=0, dtype=str)

    df_merged = df_iop1.merge(df_iop2, left_index=True, right_index=True, how='inner', suffixes=('_iop1', '_iop2'))
    display(df_merged)
else:
    print('One or both files are missing. Please check the file paths or generate them first.')
    df_merged = None


Both files exist, proceeding with merge.


,Q255,Q267,Q267L,Q272_iop1,Q272L,Q272R,F309,Q257,Q272_iop2,Q293
N,499,314,211,565,184,222,464,219,220,255
R2,0.83,0.72,0.73,0.83,0.72,0.81,0.85,0.90,0.91,0.87
MAE,0.84,0.84,0.91,0.82,0.91,0.94,0.85,0.47,0.59,0.52
MBE,-0.23,-0.23,-0.20,-0.32,-0.14,-0.43,-0.74,-0.00,-0.50,-0.22
RMSE,1.12,1.03,1.07,1.10,1.18,1.12,1.05,0.60,0.78,0.66


In [9]:
if df_merged is not None:
    latex_table = df_merged.to_latex()

    latex_table = latex_table.replace(r'_iop1', r'').replace(r'_iop2', r'')

    latex_table = latex_table.replace(r'N ', r'\# of paired obs.$^*$ ($N$) ')
    latex_table = latex_table.replace(r'R2 ', r'coefficient of determination (R$^2$) ')
    latex_table = latex_table.replace(r'MAE ', r'mean absolute error (MAE) in K ')
    latex_table = latex_table.replace(r'MBE ', r'mean bias error (MBE) in K ')
    latex_table = latex_table.replace(r'RMSE ', r'root mean square error (RMSE) in K ')

    print(latex_table)

\begin{tabular}{lllllllllll}
\toprule
 & Q255 & Q267 & Q267L & Q272 & Q272L & Q272R & F309 & Q257 & Q272 & Q293 \\
\midrule
\# of paired obs.$^*$ ($N$) & 499 & 314 & 211 & 565 & 184 & 222 & 464 & 219 & 220 & 255 \\
coefficient of determination (R$^2$) & 0.83 & 0.72 & 0.73 & 0.83 & 0.72 & 0.81 & 0.85 & 0.90 & 0.91 & 0.87 \\
mean absolute error (MAE) in K & 0.84 & 0.84 & 0.91 & 0.82 & 0.91 & 0.94 & 0.85 & 0.47 & 0.59 & 0.52 \\
mean bias error (MBE) in K & -0.23 & -0.23 & -0.20 & -0.32 & -0.14 & -0.43 & -0.74 & -0.00 & -0.50 & -0.22 \\
root mean square error (RMSE) in K & 1.12 & 1.03 & 1.07 & 1.10 & 1.18 & 1.12 & 1.05 & 0.60 & 0.78 & 0.66 \\
\bottomrule
\end{tabular}

